# DATA PREPARATION

In [1]:
import subprocess, sys, os, warnings
warnings.filterwarnings("ignore")

MARKER = "/kaggle/working/.cell0_pytorch_ready"

def install_t4_pytorch():
    subprocess.check_call([
        sys.executable, "-m", "pip", "install","torch==2.5.1", "torchvision==0.20.1","--index-url", "https://download.pytorch.org/whl/cu124","-q", "--force-reinstall",])

def cuda_smoke():
    import torch
    if not torch.cuda.is_available():
        return False
    try:
        t = torch.tensor([0, 1, -100], device="cuda")
        return int((t.ne(-100)).sum().item()) == 2
    except Exception:
        return False

if not os.path.isfile(MARKER):
    import torch
    print("GPU check")
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")
    print("PyTorch:", torch.__version__)
    if cuda_smoke():
        print("CUDA: OK")
        open(MARKER, "w").write("ok")
    else:
        install_t4_pytorch()
        open(MARKER, "w").write("restart")
        raise SystemExit

import torch
import torchvision
if not cuda_smoke():
    if os.path.isfile(MARKER):
        os.remove(MARKER)
    raise RuntimeError

print("GPU:", torch.cuda.get_device_name(0))
print("PyTorch:", torch.__version__)
print("CUDA: OK")

try:
    import fsspec
    fsspec_ver = fsspec.__version__
except ImportError:
    fsspec_ver = None

with open("/kaggle/working/pip-constraints.txt", "w") as f:
    f.write("torch==" + torch.__version__ + "\n")
    f.write("torchvision==" + torchvision.__version__ + "\n")
    if fsspec_ver:
        f.write("fsspec==" + fsspec_ver + "\n")

pkgs = ["roboflow", "scikit-learn"]
subprocess.check_call([
    sys.executable, "-m", "pip", "install", *pkgs,"-c", "/kaggle/working/pip-constraints.txt", "-q","--disable-pip-version-check","--upgrade-strategy", "only-if-needed",])
print("Done")

GPU check
GPU: Tesla T4
PyTorch: 2.10.0+cu128
CUDA: OK
GPU: Tesla T4
PyTorch: 2.10.0+cu128
CUDA: OK
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 249.2/249.2 kB 6.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 35.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 65.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 105.9 MB/s eta 0:00:00
Done


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.25.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2026.2.0 which is incompatible.


In [2]:
import json
import os

Experiment_Name = "Exp1_deitSmallDistilled_t05"
dataset_version = 2
SEED = 42

config = {
    "Experiment_Name": Experiment_Name,
    "dataset_version": dataset_version,
    "deit_variant": "facebook/deit-small-distilled-patch16-224",
    "Confidence_Threshold": 0.5,
    "epochs_deit": 30,
    "epochs_rcnn": 20,
    "batch_size": 16,
    "lr_deit": 2e-5,
    "lr_rcnn": 0.005,
    "weight_decay": 0.01,
    "PATIENCE": 5,
    "SEED": SEED,
    "BASELINE_ACC": 91.78,
    "dataset_path": "/kaggle/working/dataset",
    "save_dir": "/kaggle/working/results",
    "kaggle_input_dataset": "/kaggle/input/datasets/nikachuu/data-prep",}

os.makedirs(config["save_dir"], exist_ok=True)
with open("/kaggle/working/config.json", "w") as f:
    json.dump(config, f, indent=2)
print("Config saved:", "/kaggle/working/config.json")


Config saved: /kaggle/working/config.json


In [3]:
import json, os, shutil, glob
from sklearn.model_selection import train_test_split
from kaggle_secrets import UserSecretsClient
from roboflow import Roboflow

with open("/kaggle/working/config.json") as f:
    config = json.load(f)
SEED = config["SEED"]
dataset_version = config["dataset_version"]

secrets = UserSecretsClient()
try:
    rf_api_key = secrets.get_secret("rf_api_key")
except Exception:
    rf_api_key = None

if not rf_api_key:
    raise ValueError("Add Kaggle secret 'rf_api_key'")

rf = Roboflow(api_key=rf_api_key)
project = rf.workspace("nka-d7idh").project("find-infected-and-healthy")
dataset = project.version(dataset_version).download("coco")
RAW_DIR = dataset.location

print("Downloaded version", dataset_version)
print("Raw dir:", RAW_DIR)

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Find-infected-and-healthy-2 in coco:: 100%|██████████| 468/468 [00:00<00:00, 6737.44it/s]

Downloaded version 2
Raw dir: /kaggle/working/Find-infected-and-healthy-2


In [4]:
import json, os, shutil, glob
from sklearn.model_selection import train_test_split

with open("/kaggle/working/config.json") as f:
    config = json.load(f)
SEED = config["SEED"]
destination_root = config["dataset_path"]

ann_files = glob.glob(RAW_DIR + "/**/_annotations.coco.json", recursive=True)
all_images, all_annotations, categories = [], [], None

for ann_file in ann_files:
    split_dir = os.path.dirname(ann_file)
    with open(ann_file) as f:
        coco = json.load(f)
    if categories is None:
        categories = coco["categories"]
    id_offset = len(all_images)
    id_map = {}
    for img in coco["images"]:
        new_id = img["id"] + id_offset
        id_map[img["id"]] = new_id
        ni = img.copy()
        ni["id"] = new_id
        ni["_src_dir"] = split_dir
        all_images.append(ni)
    for ann in coco["annotations"]:
        na = ann.copy()
        na["id"] = ann["id"] + id_offset
        na["image_id"] = id_map[ann["image_id"]]
        all_annotations.append(na)

cat_names = {c["id"]: c["name"].lower() for c in categories}

ann_map = {}
for ann in all_annotations:
    ann_map.setdefault(ann["image_id"], []).append(ann)

strat_labels, valid_images = [], []
for img in all_images:
    anns = ann_map.get(img["id"], [])
    if not anns:
        continue
    cid = anns[0]["category_id"]
    strat_labels.append(1 if "infected" in cat_names.get(cid, "") else 0)
    valid_images.append(img)

train_imgs, temp_imgs, train_lbls, temp_lbls = train_test_split(
    valid_images, strat_labels, test_size=0.30, stratify=strat_labels, random_state=SEED)
valid_imgs, test_imgs, valid_lbls, test_lbls = train_test_split(
    temp_imgs, temp_lbls, test_size=0.50, stratify=temp_lbls, random_state=SEED)

print("Split       Images  Healthy  Infected")
print("----------------------------------------")

def save_split(name, imgs):
    d = os.path.join(destination_root, name)
    os.makedirs(d, exist_ok=True)
    saved_imgs, saved_anns = [], []
    for img in imgs:
        src_dir = img.pop("_src_dir")
        src = os.path.join(src_dir, img["file_name"])
        dst = os.path.join(d, img["file_name"])
        if os.path.exists(src):
            shutil.copy(src, dst)
            saved_imgs.append(img)
            saved_anns.extend(ann_map.get(img["id"], []))
            
    with open(os.path.join(d, "_annotations.coco.json"), "w") as f:
        json.dump({"images": saved_imgs, "annotations": saved_anns, "categories": categories}, f, indent=2)
    
    inf = sum(1 for a in saved_anns if "infected" in cat_names.get(a["category_id"], ""))
    print(f"{name:12}{len(saved_imgs):8}{len(saved_imgs)-inf:9}{inf:9}")

for name, imgs in [("train", train_imgs), ("valid", valid_imgs), ("test", test_imgs)]:
    save_split(name, imgs)

config["train_img_dir"] = os.path.join(destination_root, "train")
config["valid_img_dir"] = os.path.join(destination_root, "valid")
config["test_img_dir"] = os.path.join(destination_root, "test")
config["train_ann"] = os.path.join(destination_root, "train", "_annotations.coco.json")
config["valid_ann"] = os.path.join(destination_root, "valid", "_annotations.coco.json")
config["test_ann"] = os.path.join(destination_root, "test", "_annotations.coco.json")

with open("/kaggle/working/config.json", "w") as f:
    json.dump(config, f, indent=2)

print("")
print("Data QA")
print("Split       Images  Healthy  Infected")
print("----------------------------------------")

for split, d, ann in [
    ("train", config["train_img_dir"], config["train_ann"]),
    ("valid", config["valid_img_dir"], config["valid_ann"]),
    ("test", config["test_img_dir"], config["test_ann"]),]:
    n_img = len([f for f in os.listdir(d) if f.lower().endswith((".jpg", ".jpeg", ".png"))])
    with open(ann) as f:
        data = json.load(f)
        
    cats = {c["id"]: c["name"].lower() for c in data["categories"]}
    
    qa_img_to_cat = {}
    for a in data["annotations"]:
        if a["image_id"] not in qa_img_to_cat:
            qa_img_to_cat[a["image_id"]] = cats.get(a["category_id"], "unknown")
            
    counts = {"healthy": 0, "infected": 0}
    for img in data["images"]:
        img_cat = qa_img_to_cat.get(img["id"], "unknown")
        if "infected" in img_cat:
            counts["infected"] += 1
        elif "healthy" in img_cat:
            counts["healthy"] += 1
            
    print(f"{split:12}{n_img:8}{counts['healthy']:9}{counts['infected']:9}")

print("")
print("Done. Publish Kaggle dataset: config.json + dataset folder")
print("Input path: /kaggle/input/datasets/nikachuu/data-prep")

Split       Images  Healthy  Infected
----------------------------------------
train            324      189      135
valid             69       40       29
test              70       42       28

Data QA
Split       Images  Healthy  Infected
----------------------------------------
train            324      196      127
valid             69       42       27
test              70       43       27

Done. Publish Kaggle dataset: config.json + dataset folder
Input path: /kaggle/input/datasets/nikachuu/data-prep
